# Complement Naive Bayes - Day 5 (From Scratch)

## What is Complement Naive Bayes (CNB)?

Complement Naive Bayes (CNB) is a variant of the Naive Bayes algorithm that is specifically designed to improve classification performance on imbalanced datasets and text classification tasks. It modifies the way probabilities are estimated to reduce bias towards majority classes, making it more suitable than the standard Multinomial Naive Bayes in many cases.

---

## Challenge of Unbalanced Datasets

An unbalanced dataset means one type of data appears much more often than the other.

**Example:**
- 95% cases are "not fraud" and only 5% are "fraud"
- A model that always predicts "not fraud" will be 95% accurate but will miss all fraud cases
- This shows why special methods are needed to deal with such uneven data

---

## Working of CNB

1. For each class, compute the complement frequency: the frequency of features in all other classes combined
2. Estimate the conditional probabilities using these complement frequencies
3. Normalize the values to ensure they form valid probability distributions
4. Classify a sample by choosing the class with the maximum posterior probability

---

## CNB Formula

$$P(f|c) = \frac{count(f, \bar{c}) + \alpha}{\sum_{f'} count(f', \bar{c}) + \alpha \cdot |V|}$$

Where:
- $count(f, \bar{c})$ = count of feature f in the complement of class c
- $\alpha$ = smoothing parameter (Laplace smoothing)
- $|V|$ = vocabulary size

---

## When to Use CNB

| Scenario | Why CNB is Suitable |
|----------|---------------------|
| Imbalanced class distributions | Complement approach ensures minority classes receive fairer parameter estimates |
| Text classification | Handles discrete feature counts (word frequencies) effectively |
| Large feature spaces | Computationally efficient and easy to interpret |

---

## Limitations

- Assumes feature independence, which may reduce accuracy in real data
- Works best with discrete features like word counts; continuous data needs preprocessing
- May introduce bias in already balanced datasets

---

## One-Line Summary

**Complement Naive Bayes uses complement class frequencies to handle imbalanced datasets effectively.**

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

print("="*50)
print("COMPLEMENT NAIVE BAYES DAY 5 - FROM SCRATCH")
print("="*50)

## Complement Naive Bayes Implementation from Scratch

In [ ]:
class ComplementNaiveBayes:
    """
    Complement Naive Bayes Classifier from Scratch
    """
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = None
        self.class_priors = {}
        self.feature_probs = {}
        self.vocabulary_size = 0
    
    def fit(self, X, y):
        """
        Train the Complement Naive Bayes classifier
        """
        self.classes = np.unique(y)
        n_samples = len(y)
        self.vocabulary_size = X.shape[1]
        
        # Calculate class priors
        for cls in self.classes:
            self.class_priors[cls] = np.sum(y == cls) / n_samples
        
        # Calculate complement feature probabilities for each class
        for cls in self.classes:
            # Get complement of class (all other classes)
            complement_mask = y != cls
            X_complement = X[complement_mask]
            
            # Total word count in complement
            total_complement_words = np.sum(X_complement, axis=0).sum()
            
            # Sum of word counts per feature in complement
            feature_counts = np.sum(X_complement, axis=0)
            
            # Laplace smoothing
            smoothed_probs = (feature_counts + self.alpha) / (total_complement_words + self.alpha * self.vocabulary_size)
            self.feature_probs[cls] = smoothed_probs
    
    def predict_single(self, x):
        """
        Predict a single sample
        """
        posteriors = {}
        
        for cls in self.classes:
            # Start with log prior
            posterior = np.log(self.class_priors[cls])
            
            # Multiply by complement feature probabilities
            for i in range(len(x)):
                if x[i] > 0:
                    posterior += x[i] * np.log(self.feature_probs[cls][i])
            
            posteriors[cls] = posterior
        
        return max(posteriors, key=posteriors.get)
    
    def predict(self, X):
        return np.array([self.predict_single(x) for x in X])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

## Example: Apples vs Bananas Classification

In [ ]:
# Create dataset: Apples vs Bananas
data = {
    'Round': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    'Red': [1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1],
    'Soft': [1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1],
    'Class': ['Apple', 'Apple', 'Apple', 'Apple', 'Apple', 'Apple', 'Apple', 'Apple',
              'Banana', 'Banana', 'Banana', 'Banana', 'Banana', 'Banana', 'Banana', 'Banana']
}

df = pd.DataFrame(data)
print("Dataset:")
print(df)

In [ ]:
# Prepare features and target
X = df[['Round', 'Red', 'Soft']].values
y = np.array([0 if label == 'Apple' else 1 for label in df['Class']])

print("Feature matrix shape:", X.shape)
print("Class distribution:")
print(f"Apples: {np.sum(y == 0)}")
print(f"Bananas: {np.sum(y == 1)}")

In [ ]:
# Train Complement Naive Bayes
cnb = ComplementNaiveBayes(alpha=1.0)
cnb.fit(X, y)

print("\nClass Priors:")
for cls in cnb.classes:
    class_name = "Apple" if cls == 0 else "Banana"
    print(f"P({class_name}) = {cnb.class_priors[cls]:.4f}")

In [ ]:
# Display complement feature probabilities
print("\nComplement Feature Probabilities:")
features = ['Round', 'Red', 'Soft']
for cls in cnb.classes:
    class_name = "Apple" if cls == 0 else "Banana"
    print(f"\nFor {class_name} (using complement class):")
    for i, feature in enumerate(features):
        print(f"  P({feature}=1) = {cnb.feature_probs[cls][i]:.4f}")

In [ ]:
# Test new sample
test_sample = np.array([[1, 1, 1]])  # Round=1, Red=1, Soft=1
prediction = cnb.predict_single(test_sample[0])

print("\n" + "="*40)
print(f"Test Sample: Round=1, Red=1, Soft=1")
print(f"Prediction: {'Apple' if prediction == 0 else 'Banana'}")

In [ ]:
# Test multiple samples
test_samples = [
    [1, 1, 1],
    [1, 0, 0],
    [0, 1, 1],
    [0, 0, 0]
]

print("\n" + "="*40)
print("Multiple Test Samples:")
print("="*40)

for sample in test_samples:
    pred = cnb.predict_single(np.array(sample))
    class_name = "Apple" if pred == 0 else "Banana"
    print(f"Round={sample[0]}, Red={sample[1]}, Soft={sample[2]} -> {class_name}")

In [ ]:
# Compare with Multinomial Naive Bayes
class MultinomialNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = None
        self.class_priors = {}
        self.feature_probs = {}
        self.vocabulary_size = 0
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        n_samples = len(y)
        self.vocabulary_size = X.shape[1]
        
        for cls in self.classes:
            self.class_priors[cls] = np.sum(y == cls) / n_samples
            X_cls = X[y == cls]
            total_words = np.sum(X_cls, axis=0).sum()
            feature_counts = np.sum(X_cls, axis=0)
            self.feature_probs[cls] = (feature_counts + self.alpha) / (total_words + self.alpha * self.vocabulary_size)
    
    def predict_single(self, x):
        posteriors = {}
        for cls in self.classes:
            posterior = np.log(self.class_priors[cls])
            for i in range(len(x)):
                if x[i] > 0:
                    posterior += x[i] * np.log(self.feature_probs[cls][i])
            posteriors[cls] = posterior
        return max(posteriors, key=posteriors.get)
    
    def predict(self, X):
        return np.array([self.predict_single(x) for x in X])

# Compare both models
mnb = MultinomialNaiveBayes(alpha=1.0)
mnb.fit(X, y)

print("\n" + "="*50)
print("CNB vs Multinomial NB Comparison")
print("="*50)

test_sample = np.array([[1, 1, 1]])
cnb_pred = cnb.predict(test_sample)[0]
mnb_pred = mnb.predict(test_sample)[0]

print(f"Test: Round=1, Red=1, Soft=1")
print(f"Complement NB: {'Apple' if cnb_pred == 0 else 'Banana'}")
print(f"Multinomial NB: {'Apple' if mnb_pred == 0 else 'Banana'}")

In [ ]:
# Day 5 Completed
print("\n" + "="*50)
print("COMPLEMENT NAIVE BAYES DAY 5 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Complement Naive Bayes definition")
print("- Challenge of unbalanced datasets")
print("- Working of CNB (complement frequencies)")
print("- CNB formula")
print("- Apples vs Bananas example")
print("- CNB vs Multinomial NB comparison")
print("- When to use CNB")
print("- Limitations of CNB")
print("="*50)"
print("\n*** NAIVE BAYES MODULE COMPLETED ***")
print("\nComplete Naive Bayes Series:")
print("Day 1: Introduction to Naive Bayes")
print("Day 2: Gaussian Naive Bayes")
print("Day 3: Multinomial Naive Bayes")
print("Day 4: Bernoulli Naive Bayes")
print("Day 5: Complement Naive Bayes (Final)")